<a href="https://colab.research.google.com/github/isaiahdm792-alt/stock--screener/blob/main/01_full_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import requests
from io import StringIO # Import StringIO

def get_sp500_tickers():
    """Scrape the current S&P 500 constituent list from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

    # Wrap the response text in StringIO as suggested by the FutureWarning
    tables = pd.read_html(StringIO(response.text))
    sp500_table = tables[0]  # first table on the page is the constituent list
    tickers = sp500_table["Symbol"].tolist()
    # yfinance expects dots as hyphens for some tickers (e.g. BRK.B -> BRK-B)
    tickers = [t.replace(".", "-") for t in tickers]
    return tickers

sp500_tickers = get_sp500_tickers()
print(f"Pulled {len(sp500_tickers)} tickers.")
print(sp500_tickers[:10], "...")

Pulled 503 tickers.
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A'] ...


In [6]:
# STEP 1: Fundamentals Scanner — starter script
# Paste this into a Google Colab cell and run it.
# First run: install yfinance (only needed once per Colab session)

# !pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import time

# --- Start small: 15 well-known tickers to test the pipeline first ---
# Once this works cleanly, swap in the full S&P 500 list (step below).
test_tickers = sp500_tickers

def get_fundamentals(ticker):
    """Pull key fundamental ratios for one ticker. Returns a dict or None on failure."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        pe = info.get("trailingPE")
        ps = info.get("priceToSalesTrailing12Months")
        pb = info.get("priceToBook")
        peg = info.get("pegRatio")
        fcf = info.get("freeCashflow")
        earnings_growth = info.get("earningsGrowth")
        sector = info.get("sector")

        return {
            "ticker": ticker,
            "sector": sector,
            "pe_ratio": pe,
            "ps_ratio": ps,
            "pb_ratio": pb,
            "peg_ratio": peg,
            "free_cash_flow": fcf,
            "earnings_growth": earnings_growth,
        }
    except Exception as e:
        print(f"  [!] Failed on {ticker}: {e}")
        return None

# --- Run the scan ---
results = []
for t in test_tickers:
    print(f"Pulling {t}...")
    data = get_fundamentals(t)
    if data:
        results.append(data)
    time.sleep(0.5)  # be polite to the API, avoid rate-limit issues

df = pd.DataFrame(results)

# --- Basic cleanup: drop rows missing the core ratios ---
df_clean = df.dropna(subset=["pe_ratio", "ps_ratio"])

print("\n--- Results ---")
print(df_clean.sort_values("pe_ratio").to_string(index=False))

# --- Save to CSV so you can commit it to GitHub / inspect later ---
df_clean.to_csv("fundamentals_snapshot.csv", index=False)
print("\nSaved to fundamentals_snapshot.csv")

Pulling MMM...
Pulling AOS...
Pulling ABT...
Pulling ABBV...
Pulling ACN...
Pulling ADBE...
Pulling AMD...
Pulling AES...
Pulling AFL...
Pulling A...
Pulling APD...
Pulling ABNB...
Pulling AKAM...
Pulling ALB...
Pulling ARE...
Pulling ALGN...
Pulling ALLE...
Pulling LNT...
Pulling ALL...
Pulling GOOGL...
Pulling GOOG...
Pulling MO...
Pulling AMZN...
Pulling AMCR...
Pulling AEE...
Pulling AEP...
Pulling AXP...
Pulling AIG...
Pulling AMT...
Pulling AWK...
Pulling AMP...
Pulling AME...
Pulling AMGN...
Pulling APH...
Pulling ADI...
Pulling AON...
Pulling APA...
Pulling APO...
Pulling AAPL...
Pulling AMAT...
Pulling APP...
Pulling APTV...
Pulling ACGL...
Pulling ADM...
Pulling ARES...
Pulling ANET...
Pulling AJG...
Pulling AIZ...
Pulling T...
Pulling ATO...
Pulling ADSK...
Pulling ADP...
Pulling AZO...
Pulling AVB...
Pulling AVY...
Pulling AXON...
Pulling BKR...
Pulling BALL...
Pulling BAC...
Pulling BAX...
Pulling BDX...
Pulling BRK-B...
Pulling BBY...
Pulling TECH...
Pulling BIIB...
Pulli

In [7]:
# STEP 2: Basic sector-relative scoring
import pandas as pd

df = df_clean.copy()  # use the results from Step 1

# Calculate the average P/E per sector
sector_avg_pe = df.groupby("sector")["pe_ratio"].transform("mean")

# Flag stocks trading below their sector average P/E
df["pe_vs_sector"] = df["pe_ratio"] - sector_avg_pe
df["undervalued_flag"] = df["pe_vs_sector"] < 0

# Simple composite score: lower PEG and lower relative P/E = higher score
df["score"] = (
    (1 / df["peg_ratio"].clip(lower=0.1)) * 40   # reward low PEG
    - df["pe_vs_sector"].clip(lower=0) * 0.5      # penalize high relative P/E
)

df_ranked = df.sort_values("score", ascending=False)

print(df_ranked[["ticker", "sector", "pe_ratio", "peg_ratio", "undervalued_flag", "score"]].head(20).to_string(index=False))

df_ranked.to_csv("scored_snapshot.csv", index=False)
print("\nSaved to scored_snapshot.csv")

ticker             sector   pe_ratio  peg_ratio  undervalued_flag      score
   HIG Financial Services  10.060773       0.12              True 333.333333
  CSGP        Real Estate 165.722210       0.11             False 308.490717
   COF Financial Services  11.492842       0.22              True 181.818182
   FIS         Technology   9.071706       0.24              True 166.666667
    SW  Consumer Cyclical  51.340424       0.26             False 152.203787
    ON         Technology  57.985294       0.29              True 137.931034
   TPR  Consumer Cyclical  45.626140       0.30              True 133.333333
   CVS         Healthcare  46.456140       0.30             False 126.711796
    GM  Consumer Cyclical  39.910713       0.35              True 114.285714
   LUV        Industrials  29.006536       0.36              True 111.111111
   BEN Financial Services  24.725191       0.37             False 105.981936
   EME        Industrials  22.574017       0.40              True 100.000000